# 00 · Control de integridad del archivo

## Objetivo
Comparar la copia de trabajo de este expediente con el archivo fuente, byte a byte, antes de interpretar hojas o fórmulas.

## Alcance
El control cubre tres rutas del mismo libro BES-jun: la copia `entrada/BES_jun.xlsx`, la copia preservada en `fuentes/originales/` y el archivo recibido en la raíz del proyecto. No abre el libro, no lee otros Excel y no evalúa mediciones ni ecuaciones.


In [1]:
from pathlib import Path
import os, sys
aqui = Path.cwd().resolve()
expediente = next(c for c in [aqui, *aqui.parents] if (c / "dataset.md").is_file() and (c / "entrada").is_dir())
os.chdir(expediente)
if str(expediente / "src") not in sys.path:
    sys.path.insert(0, str(expediente / "src"))
libro = expediente / "entrada" / "BES_jun.xlsx"

import hashlib
proyecto = next(c for c in expediente.parents if (c / "fuentes" / "originales").is_dir() and (c / "trabajo.md").is_file())
rutas = [
    libro,
    proyecto / "fuentes" / "originales" / "Copia de FINAL BES-jun.xlsx",
    proyecto / "Copia de FINAL BES-jun.xlsx",
]
registros = []
for ruta in rutas:
    datos = ruta.read_bytes()
    registros.append((str(ruta.relative_to(proyecto)), len(datos), hashlib.sha256(datos).hexdigest(), hashlib.md5(datos).hexdigest()))
for ruta, n, sha, md5 in registros:
    print(f"ruta={ruta}")
    print(f"bytes={n}")
    print(f"sha256={sha}")
    print(f"md5={md5}")
    print("---")
coinciden = len({r[1] for r in registros}) == 1 and len({r[2] for r in registros}) == 1
print("coincidencia_binaria=" + ("si" if coinciden else "no"))
assert coinciden
assert registros[0][2] == "ccb5a738baea13321993e8b6eccefd6cac90ecda86b1c4b9181f897fcfa83f85"


ruta=datasets/en_uso/BES-jun/entrada/BES_jun.xlsx
bytes=329308
sha256=ccb5a738baea13321993e8b6eccefd6cac90ecda86b1c4b9181f897fcfa83f85
md5=ba89f317f505f321c496e44ab5046104
---
ruta=fuentes/originales/Copia de FINAL BES-jun.xlsx
bytes=329308
sha256=ccb5a738baea13321993e8b6eccefd6cac90ecda86b1c4b9181f897fcfa83f85
md5=ba89f317f505f321c496e44ab5046104
---
ruta=Copia de FINAL BES-jun.xlsx
bytes=329308
sha256=ccb5a738baea13321993e8b6eccefd6cac90ecda86b1c4b9181f897fcfa83f85
md5=ba89f317f505f321c496e44ab5046104
---
coincidencia_binaria=si


## Qué demuestra la salida
Cada bloque imprime la ruta relativa a la raíz del proyecto, el tamaño en bytes, el SHA-256 completo y el MD5. El indicador `coincidencia_binaria=si` exige que los tres tamaños sean iguales y que los tres SHA-256 sean iguales. El valor esperado de ese SHA-256, registrado en `dataset.md`, es `ccb5a738baea13321993e8b6eccefd6cac90ecda86b1c4b9181f897fcfa83f85`, con 329308 bytes.

## Qué no demuestra
La identidad binaria confirma que el análisis parte del mismo archivo que se archivó. No confirma que las mediciones sean correctas, que las unidades estén bien rotuladas ni que las fórmulas del libro sean adecuadas.
